# Chapter 2 — End-to-End Machine Learning Project
Companion notebook to `part1_fundamentals/02_end_to_end_project.md`.

Uses scikit-learn's built-in `fetch_california_housing` (a close cousin of the book's
original `housing.csv`, minus the categorical `ocean_proximity` column). Needs internet
access the first time, to download the dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

## 1-2. Get the data & look at it

In [ ]:
housing = fetch_california_housing(as_frame=True).frame
housing = housing.rename(columns={"MedHouseVal": "median_house_value"})
housing.head()

In [ ]:
housing.info()
housing.describe()

In [ ]:
housing.hist(bins=50, figsize=(12, 8))
plt.tight_layout()
plt.show()

## Stratified train/test split (stratify on income, the most predictive feature)

In [ ]:
housing["income_cat"] = pd.cut(housing["MedInc"],
                                bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                                labels=[1, 2, 3, 4, 5])

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.loc[train_index]
    strat_test_set = housing.loc[test_index]

for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

print(len(strat_train_set), "train +", len(strat_test_set), "test")

## 3. Explore (training set only!)

In [ ]:
housing_expl = strat_train_set.copy()
housing_expl.plot(kind="scatter", x="Longitude", y="Latitude", alpha=0.4,
                   s=housing_expl["Population"] / 100, label="population", figsize=(10, 7),
                   c="median_house_value", cmap="jet", colorbar=True)
plt.legend()
plt.show()

In [ ]:
corr_matrix = housing_expl.corr(numeric_only=True)
corr_matrix["median_house_value"].sort_values(ascending=False)

In [ ]:
# Feature engineering: ratios often beat raw counts
housing_expl["rooms_per_household"] = housing_expl["AveRooms"] / housing_expl["AveOccup"]
housing_expl.corr(numeric_only=True)["median_house_value"].sort_values(ascending=False)

## 4. Prepare the data (reusable pipeline)

In [ ]:
housing_tr = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

num_attribs = list(housing_tr)   # all columns are numeric in this dataset

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("std_scaler", StandardScaler()),
])

full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
])

housing_prepared = full_pipeline.fit_transform(housing_tr)
housing_prepared.shape

## 5. Select & train a model, compare with cross-validation

In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(housing_prepared, housing_labels)
lin_rmse = mean_squared_error(housing_labels, lin_reg.predict(housing_prepared), squared=False)
print("Linear Regression training RMSE:", lin_rmse)

In [ ]:
for name, model in [("Linear Regression", LinearRegression()),
                     ("Decision Tree", DecisionTreeRegressor(random_state=42)),
                     ("Random Forest", RandomForestRegressor(n_estimators=100, random_state=42))]:
    scores = cross_val_score(model, housing_prepared, housing_labels,
                              scoring="neg_mean_squared_error", cv=10)
    rmse_scores = np.sqrt(-scores)
    print(f"{name:18s} mean RMSE = {rmse_scores.mean():.0f}  std = {rmse_scores.std():.0f}")

## 6. Fine-tune with GridSearchCV

In [ ]:
param_grid = [
    {"n_estimators": [30, 100], "max_features": [4, 6, 8]},
]
forest_reg = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(forest_reg, param_grid, cv=5,
                            scoring="neg_mean_squared_error", return_train_score=True)
grid_search.fit(housing_prepared, housing_labels)
print(grid_search.best_params_)
best_model = grid_search.best_estimator_

In [ ]:
# Which features matter most?
importances = best_model.feature_importances_
sorted(zip(importances, num_attribs), reverse=True)

## Final evaluation — touch the test set exactly once

In [ ]:
X_test = strat_test_set.drop("median_house_value", axis=1)
y_test = strat_test_set["median_house_value"].copy()

X_test_prepared = full_pipeline.transform(X_test)   # transform only, never fit again!
final_predictions = best_model.predict(X_test_prepared)
final_rmse = mean_squared_error(y_test, final_predictions, squared=False)
print("Final test RMSE:", final_rmse)

## ✍️ Practice
1. Compare Linear Regression, Decision Tree, and Random Forest 10-fold CV RMSE — which wins, by how much?
2. Explain why fitting a `StandardScaler` on train+test combined is a mistake.
3. Your `DecisionTreeRegressor` gets RMSE=0 on train but poor CV score — what's happening, and what's one fix?
4. Design: if predictions only need to feed a "cheap/medium/expensive" downstream system, how would framing change?

In [ ]:
# Your code / notes here